# Numerikus módszerek -- 5. előadás + gyakorlat
## Ortogonális mátrixok, QR-felbontás, Gram–Schmidt, Householder

Ez a notebook **részletes kidolgozott példákat** és **önálló feladatokat** tartalmaz a következő témákhoz:

1. Ortogonális mátrixok (definíció, tulajdonságok, példák)
2. QR-felbontás (definíció, létezés, egyértelműség)
3. Gram–Schmidt-féle ortogonalizáció
4. Householder-féle mátrixok és transzformációk
5. Householder-transzformációk alkalmazásai
6. Önálló feladatok
7. Debuggolási feladatok

In [ ]:
import numpy as np
from scipy.linalg import qr as scipy_qr
import time

np.set_printoptions(precision=4, suppress=True)

def print_matrix(M, label=""):
    """Mátrix szép kiírása."""
    if label:
        print(label)
    for i in range(M.shape[0]):
        row = "  ".join(f"{M[i,j]:8.4f}" for j in range(M.shape[1]))
        print(f"  [{row}]")
    print()

def print_vector(v, label=""):
    """Vektor szép kiírása."""
    if label:
        print(f"{label} = [{', '.join(f'{x:.4f}' for x in v)}]")

---
## 1. Ortogonális mátrixok

### Definíció

Egy $Q \in \mathbb{R}^{n \times n}$ mátrix **ortogonális**, ha az inverze a transzponáltja, azaz:

$$Q^\top Q = I$$

Ekkor $QQ^\top = I$ is teljesül, tehát $Q^{-1} = Q^\top$.

### Skaláris szorzat

Az $x, y \in \mathbb{R}^n$ vektorok **skaláris szorzata**:

$$\langle x, y \rangle := y^\top x = \sum_{k=1}^{n} x_k \cdot y_k$$

### Ortonormált rendszer

A $q_1, \ldots, q_n \in \mathbb{R}^n$ vektorok **ortonormált rendszert** alkotnak, ha:

$$\langle q_i, q_j \rangle = \begin{cases} 0 & \text{ha } i \neq j, \\ 1 & \text{ha } i = j. \end{cases} = \delta_{ij} \quad \text{(Kronecker-féle delta)}$$

**Állítás:** Egy $Q \in \mathbb{R}^{n \times n}$ ortogonális mátrix oszlopai, mint vektorok ortonormált rendszert alkotnak.

### Tulajdonságok
- Ortogonális mátrixok szorzata is ortogonális: $(Q_1 Q_2)^\top(Q_1 Q_2) = Q_2^\top Q_1^\top Q_1 Q_2 = I$
- $\det(Q) = \pm 1$
- Hosszmegőrző: $\|Qx\|_2 = \|x\|_2$

In [ ]:
# === Példák ortogonális mátrixokra ===
print("Példák ortogonális mátrixokra")
print("=" * 55)

# 1. Egységmátrix
I2 = np.eye(2)
print_matrix(I2, "\n1. Egységmátrix:")
print(f"   Q^T Q = I? {np.allclose(I2.T @ I2, np.eye(2))}")

# 2. 1/sqrt(2) * [[1, -1], [1, 1]]
Q2 = 1/np.sqrt(2) * np.array([[1, -1], [1, 1]])
print_matrix(Q2, "\n2. Q = 1/√2 · [[1, -1], [1, 1]]:")
print_matrix(Q2.T @ Q2, "   Q^T Q =")
print(f"   Q^T Q = I? {np.allclose(Q2.T @ Q2, np.eye(2))}")

# 3. Forgatómátrix (φ = π/4)
phi = np.pi / 4
Q3 = np.array([[np.cos(phi), -np.sin(phi)],
               [np.sin(phi),  np.cos(phi)]])
print_matrix(Q3, f"\n3. Forgatómátrix (φ = π/4):")
print(f"   Q^T Q = I? {np.allclose(Q3.T @ Q3, np.eye(2))}")
print(f"   det(Q) = {np.linalg.det(Q3):.4f}")

# Tulajdonságok demonstrálása
print("\n=== Tulajdonságok ===")
print(f"\nOrtogonális mátrixok szorzata ortogonális?")
Q12 = Q2 @ Q3
print(f"  (Q2 · Q3)^T (Q2 · Q3) = I? {np.allclose(Q12.T @ Q12, np.eye(2))}")

x = np.array([3.0, 4.0])
print(f"\nHosszmegőrzés: ||x||₂ = {np.linalg.norm(x):.4f}")
print(f"  ||Q2 · x||₂ = {np.linalg.norm(Q2 @ x):.4f}")
print(f"  ||Q3 · x||₂ = {np.linalg.norm(Q3 @ x):.4f}")

---
## 2. QR-felbontás

### Definíció

Az $A \in \mathbb{R}^{n \times n}$ mátrix **QR-felbontásának** nevezzük a $Q \cdot R$ szorzatot, ha $A = QR$, ahol:
- $Q \in \mathbb{R}^{n \times n}$ **ortogonális** mátrix,
- $R \in \mathcal{U}$ **felső háromszögmátrix**.

### Tétel: QR-felbontás létezése és egyértelműsége

Ha $\det A \neq 0$ (vagyis az $A$ oszlopvektorai lineárisan függetlenek), akkor $A$-nak létezik QR-felbontása.
Ha még feltesszük, hogy $r_{ii} > 0$ $\forall\, i$-re, akkor egyértelmű is.

**Biz. (Létezés):** A bizonyítást a Gram–Schmidt-féle ortogonalizációs eljárás adja.

**Biz. (Egyértelműség):** Indirekt: ha $A = Q_1 R_1 = Q_2 R_2$, akkor $Q_2^\top Q_1 = R_2 R_1^{-1}$. A bal oldal ortogonális, a jobb oldal felső háromszög → mindkettő egységmátrix → $R_1 = R_2$, $Q_1 = Q_2$.

### Miért jó a QR-felbontás?

Ha az $Ax = b$ LER-t kell megoldanunk és rendelkezésünkre áll az $A = QR$ felbontás:

$$Ax = b \quad \Longleftrightarrow \quad QRx = b \quad \Longleftrightarrow \quad Rx = Q^\top b$$

| Lépés | Művelet | Műveletigény |
|-------|---------|-------------|
| QR-felbontás előállítása | (egyszer) | $2n^3 + \mathcal{O}(n^2)$ |
| $y = Q^\top b$ | mátrix-vektor szorzás | $2n^2 + \mathcal{O}(n)$ |
| $Rx = y$ | visszahelyettesítés | $n^2 + \mathcal{O}(n)$ |

Így numerikusan **stabilabb** a LER megoldása, mint GE-vel.

---
## 3. Gram–Schmidt-féle ortogonalizáció

### Feladat

Adott $a_1, \ldots, a_n \in \mathbb{R}^n$ lineárisan független vektorrendszer, készítsünk belőlük egy $q_1, \ldots, q_n \in \mathbb{R}^n$ ortonormált vektorrendszert úgy, hogy $q_k$ csak $a_1, \ldots, a_k$-tól függ.

Másképp, mátrixszorzás alakban: $QR = A$, azaz:

$$\begin{bmatrix} q_1 & q_2 & \cdots & q_n \end{bmatrix} \cdot \begin{bmatrix} r_{11} & r_{12} & \cdots & r_{1n} \\ & r_{22} & \cdots & r_{2n} \\ & & \ddots & \vdots \\ & & & r_{nn} \end{bmatrix} = \begin{bmatrix} a_1 & a_2 & \cdots & a_n \end{bmatrix}$$

### Algoritmus (normálással)

Adott az $a_1, \ldots, a_n \in \mathbb{R}^n$ lineárisan független vektorrendszer.

1. $r_{11} := \|a_1\|_2$
2. $q_1 := \frac{1}{r_{11}} a_1$ („lenormáljuk")

A $k$-adik lépésben ($k = 2, \ldots, n$):

3. $r_{jk} := \langle a_k, q_j \rangle$ $(j = 1, \ldots, k-1)$
4. $s_k := a_k - \sum_{j=1}^{k-1} r_{jk} \cdot q_j$ ($s_k$ segédvektor)
5. $r_{kk} := \|s_k\|_2$ (segédvektor hossza)
6. $q_k := \frac{1}{r_{kk}} s_k$ („lenormáljuk")

### Algoritmus (normálás nélkül — kézi számolásra)

1. $\tilde{q}_1 := a_1$, $\tilde{r}_{11} := 1$
2. A $k$-adik lépésben ($k = 2, \ldots, n$):
   - $\tilde{r}_{jk} := \frac{\langle a_k, \tilde{q}_j \rangle}{\langle \tilde{q}_j, \tilde{q}_j \rangle}$ $(j = 1, \ldots, k-1)$
   - $\tilde{q}_k := a_k - \sum_{j=1}^{k-1} \tilde{r}_{jk} \cdot \tilde{q}_j$

Utólag normálunk: $Q = \tilde{Q} \cdot \text{diag}(\|\tilde{q}_1\|_2^{-1}, \ldots, \|\tilde{q}_n\|_2^{-1})$, $R$ sorait szorozzuk $\|\tilde{q}_i\|_2$-vel.

### Műveletigény

A szorzások és osztások száma: $2n^3 + \mathcal{O}(n^2)$, valamint $n$ darab négyzetgyökvonás.

### Kidolgozott példa: 2×2 QR-felbontás Gram–Schmidt-tel

Készítsük el a következő mátrix QR-felbontását Gram–Schmidt-ortogonalizációval (normálással)!

$$
A = \begin{bmatrix} 1 & 2 \\ 2 & 1 \end{bmatrix}
$$

In [ ]:
A_gs = np.array([[1, 2],
                 [2, 1]], dtype=float)
print_matrix(A_gs, "A =")

a1 = A_gs[:, 0]
a2 = A_gs[:, 1]

# === 1. lépés: q1, r11 ===
print("=== 1. lépés: a1-ből q1 ===")
r11 = np.linalg.norm(a1)
print(f"  r11 = ||a1||₂ = √(1² + 2²) = √{np.dot(a1,a1):.0f} = {r11:.4f}")

q1 = a1 / r11
print(f"  q1 = a1/r11 = 1/{r11:.4f} · [{a1[0]:.0f}, {a1[1]:.0f}]ᵀ = [{q1[0]:.4f}, {q1[1]:.4f}]ᵀ")

# === 2. lépés: r12, s2, r22, q2 ===
print("\n=== 2. lépés: a2-ből q2 ===")
r12 = np.dot(a2, q1)
print(f"  r12 = ⟨a2, q1⟩ = [{a2[0]:.0f}, {a2[1]:.0f}]ᵀ · [{q1[0]:.4f}, {q1[1]:.4f}]ᵀ = 1/√5·(2·1 + 1·2) = {r12:.4f}")

s2 = a2 - r12 * q1
print(f"  s2 = a2 - r12·q1 = [{a2[0]:.0f}, {a2[1]:.0f}]ᵀ - {r12:.4f}·[{q1[0]:.4f}, {q1[1]:.4f}]ᵀ = [{s2[0]:.4f}, {s2[1]:.4f}]ᵀ")

r22 = np.linalg.norm(s2)
print(f"  r22 = ||s2||₂ = √({s2[0]:.4f}² + {s2[1]:.4f}²) = {r22:.4f}")

q2 = s2 / r22
print(f"  q2 = s2/r22 = [{q2[0]:.4f}, {q2[1]:.4f}]ᵀ")

# Eredmények
Q_gs = np.column_stack([q1, q2])
R_gs = np.array([[r11, r12],
                 [0,   r22]])

print_matrix(Q_gs, "\nQ =")
print_matrix(R_gs, "R =")

print(f"Ellenőrzés: Q @ R = A? {np.allclose(Q_gs @ R_gs, A_gs)}")
print(f"Q ortogonális? Q^T Q = I? {np.allclose(Q_gs.T @ Q_gs, np.eye(2))}")
print_matrix(Q_gs @ R_gs, "Q @ R =")

In [ ]:
def gram_schmidt(A):
    """Gram–Schmidt ortogonalizáció (klasszikus, normálással).
    
    Paraméterek:
        A: n×n invertálható mátrix
    
    Visszatérés:
        Q: ortogonális mátrix
        R: felső háromszögmátrix
    """
    n = A.shape[0]
    Q = np.zeros((n, n))
    R = np.zeros((n, n))
    
    for k in range(n):
        # Projekciók kiszámítása
        for j in range(k):
            R[j, k] = np.dot(A[:, k], Q[:, j])
        
        # Segédvektor
        s = A[:, k] - Q[:, :k] @ R[:k, k]
        
        # Normálás
        R[k, k] = np.linalg.norm(s)
        Q[:, k] = s / R[k, k]
    
    return Q, R

# Ellenőrzés a 2×2 példával
Q_check, R_check = gram_schmidt(A_gs)
print(f"Gram–Schmidt függvény ellenőrzése:")
print(f"  Q @ R = A? {np.allclose(Q_check @ R_check, A_gs)}")
print(f"  Q^T Q = I? {np.allclose(Q_check.T @ Q_check, np.eye(2))}")

### Kidolgozott példa: 3×3 QR-felbontás Gram–Schmidt-tel

$$
A = \begin{bmatrix} 1 & 1 & 0 \\ 1 & 0 & 1 \\ 0 & 1 & 1 \end{bmatrix}
$$

In [ ]:
A3 = np.array([[1, 1, 0],
               [1, 0, 1],
               [0, 1, 1]], dtype=float)
print_matrix(A3, "A =")

a1, a2, a3 = A3[:, 0], A3[:, 1], A3[:, 2]

# === 1. lépés: q1 ===
print("=== 1. lépés: a1 → q1 ===")
r11 = np.linalg.norm(a1)
q1 = a1 / r11
print(f"  r11 = ||a1||₂ = √(1² + 1² + 0²) = √2 = {r11:.4f}")
print(f"  q1 = a1/r11 = [{q1[0]:.4f}, {q1[1]:.4f}, {q1[2]:.4f}]ᵀ")

# === 2. lépés: q2 ===
print("\n=== 2. lépés: a2 → q2 ===")
r12 = np.dot(a2, q1)
print(f"  r12 = ⟨a2, q1⟩ = (1·1/√2 + 0·1/√2 + 1·0) = {r12:.4f}")
s2 = a2 - r12 * q1
print(f"  s2 = a2 - r12·q1 = [{s2[0]:.4f}, {s2[1]:.4f}, {s2[2]:.4f}]ᵀ")
r22 = np.linalg.norm(s2)
q2 = s2 / r22
print(f"  r22 = ||s2||₂ = {r22:.4f}")
print(f"  q2 = s2/r22 = [{q2[0]:.4f}, {q2[1]:.4f}, {q2[2]:.4f}]ᵀ")

# === 3. lépés: q3 ===
print("\n=== 3. lépés: a3 → q3 ===")
r13 = np.dot(a3, q1)
r23 = np.dot(a3, q2)
print(f"  r13 = ⟨a3, q1⟩ = {r13:.4f}")
print(f"  r23 = ⟨a3, q2⟩ = {r23:.4f}")
s3 = a3 - r13 * q1 - r23 * q2
print(f"  s3 = a3 - r13·q1 - r23·q2 = [{s3[0]:.4f}, {s3[1]:.4f}, {s3[2]:.4f}]ᵀ")
r33 = np.linalg.norm(s3)
q3 = s3 / r33
print(f"  r33 = ||s3||₂ = {r33:.4f}")
print(f"  q3 = s3/r33 = [{q3[0]:.4f}, {q3[1]:.4f}, {q3[2]:.4f}]ᵀ")

# Eredmények
Q3 = np.column_stack([q1, q2, q3])
R3 = np.array([[r11, r12, r13],
               [0,   r22, r23],
               [0,   0,   r33]])

print_matrix(Q3, "\nQ =")
print_matrix(R3, "R =")
print(f"Q @ R = A? {np.allclose(Q3 @ R3, A3)}")
print(f"Q ortogonális? {np.allclose(Q3.T @ Q3, np.eye(3))}")

# Összehasonlítás numpy-val
Q_np, R_np = np.linalg.qr(A3)
print(f"\nnumpy.linalg.qr egyezik? Q: {np.allclose(np.abs(Q3), np.abs(Q_np))}, R: {np.allclose(np.abs(R3), np.abs(R_np))}")
print("(Megjegyzés: az előjel eltérhet, mert numpy nem követeli meg r_ii > 0-t)")

---
## 4. Householder-féle mátrixok

### Definíció

A $H = H(v) \in \mathbb{R}^{n \times n}$ mátrixot **Householder-mátrixnak** nevezzük, ha:

$$H(v) = I - 2vv^\top,$$

ahol $v \in \mathbb{R}^n$ és $\|v\|_2 = 1$.

### Megjegyzések
- $H(v)$ transzformációs mátrixot nem kell előállítani, elég alkalmazni vektorokra:
  - $H(v)x = (I - 2vv^\top)x = x - 2v\underbrace{(v^\top x)}_{\in \mathbb{R}}$: mindössze $4n$ művelet ($2n^2 + \mathcal{O}(n)$ helyett!)
- $H(v)$ egy **tükröző** mátrix: a $v$-re merőleges $n-1$ dimenziós altérre (síkra) tükröz.

### Tulajdonságok

1. $H^\top = H$ (szimmetrikus)
2. $H^2 = I$, azaz $H^{-1} = H$ (ortogonális)
3. $H(v) \cdot v = -v$ (a normálvektort tükrözi)
4. $\forall\, y \perp v$: $H(v) \cdot y = y$ (a merőleges vektorokat megtartja)

### Tétel: tetszőleges tükrözés Householder-mátrixszal

Legyen $a, b \in \mathbb{R}^n$, $a \neq b$ és $\|a\|_2 = \|b\|_2 \neq 0$. Ekkor a

$$v = \pm \frac{a - b}{\|a - b\|_2}$$

választással $H(v) \cdot a = b$.

### Speciális eset: $a \to \sigma e_1$ alakra hozás

A numerikus stabilitás érdekében: $\sigma := -\text{sgn}(a_1) \cdot \|a\|_2$.

Ekkor $v = \frac{a - \sigma e_1}{\|a - \sigma e_1\|_2}$.

In [ ]:
# === Householder-mátrix tulajdonságainak demonstrálása ===
print("Householder-mátrix tulajdonságai")
print("=" * 55)

# Válasszunk egy v vektort
v = np.array([1, -2, 1], dtype=float)
v = v / np.linalg.norm(v)  # normálás
print(f"v = {v}  (||v||₂ = {np.linalg.norm(v):.4f})")

H = np.eye(3) - 2 * np.outer(v, v)
print_matrix(H, "\nH(v) = I - 2vvᵀ =")

# 1. Szimmetrikus
print(f"1. Szimmetrikus: Hᵀ = H? {np.allclose(H.T, H)}")

# 2. H² = I (ortogonális, involúció)
print(f"2. H² = I? {np.allclose(H @ H, np.eye(3))}")
print(f"   Hᵀ H = I? (ortogonális) {np.allclose(H.T @ H, np.eye(3))}")

# 3. H(v)·v = -v
Hv = H @ v
print(f"3. H(v)·v = {Hv}  =  -v = {-v}  → egyezik? {np.allclose(Hv, -v)}")

# 4. v-re merőleges vektorra: H(v)·y = y
y = np.array([2, 1, 0], dtype=float)
y = y - np.dot(y, v) * v  # merőlegesítés v-re
print(f"4. y ⊥ v: y = {y}, ⟨y,v⟩ = {np.dot(y,v):.10f}")
Hy = H @ y
print(f"   H(v)·y = {Hy}  → egyezik y-nal? {np.allclose(Hy, y)}")

# Hosszmegőrzés
x = np.array([3, 1, -2], dtype=float)
print(f"\nHosszmegőrzés: ||x||₂ = {np.linalg.norm(x):.4f}, ||H·x||₂ = {np.linalg.norm(H @ x):.4f}")

### Kidolgozott példa 1: Householder-féle tükrözés ($a \to b$)

Határozzuk meg azt a Householder-féle transzformációt, amely az azonos hosszúságú $a, b$ vektorhoz előállítja azt a $v$ vektort, melyre $H(v) \cdot a = b$!

$$
a = \begin{bmatrix} 2 \\ 0 \\ 1 \end{bmatrix}, \qquad b = \begin{bmatrix} 1 \\ 2 \\ 0 \end{bmatrix}
$$

In [ ]:
a = np.array([2, 0, 1], dtype=float)
b = np.array([1, 2, 0], dtype=float)

print(f"a = {a},  ||a||₂ = {np.linalg.norm(a):.4f}")
print(f"b = {b},  ||b||₂ = {np.linalg.norm(b):.4f}")
print(f"Azonos hosszúak? {np.allclose(np.linalg.norm(a), np.linalg.norm(b))}")

print("\n=== v meghatározása ===")
diff = a - b
print(f"  a - b = {diff}")
norm_diff = np.linalg.norm(diff)
print(f"  ||a - b||₂ = √(1² + (-2)² + 1²) = √{int(np.dot(diff, diff))} = {norm_diff:.4f}")

v = diff / norm_diff
print(f"  v = (a - b)/||a - b||₂ = 1/√6 · [1, -2, 1]ᵀ = {v}")

print("\n=== Ellenőrzés: H(v)·a = b? ===")
print("  H(v)·a = a - 2v(vᵀa)")
vTa = np.dot(v, a)
print(f"  vᵀa = {v[0]:.4f}·{a[0]:.0f} + {v[1]:.4f}·{a[1]:.0f} + {v[2]:.4f}·{a[2]:.0f} = {vTa:.4f}")
Ha = a - 2 * vTa * v
print(f"  H(v)·a = [{a[0]:.0f}, {a[1]:.0f}, {a[2]:.0f}]ᵀ - 2·{vTa:.4f}·[{v[0]:.4f}, {v[1]:.4f}, {v[2]:.4f}]ᵀ")
print(f"         = [{a[0]:.0f}, {a[1]:.0f}, {a[2]:.0f}]ᵀ - {2*vTa:.4f}·[{v[0]:.4f}, {v[1]:.4f}, {v[2]:.4f}]ᵀ")
print(f"         = {Ha}")
print(f"  b      = {b}")
print(f"  H(v)·a = b? {np.allclose(Ha, b)}")

### Kidolgozott példa 2: $a \to \sigma e_1$ alakra hozás

Határozzuk meg azt a Householder-transzformációt, amely a következő $a$ vektort $b = \sigma \cdot e_1$ alakúra hozza!

$$
a = \begin{bmatrix} 2 \\ -2 \\ 1 \end{bmatrix}
$$

A jó előjel választás $\sigma$-nak $-1$, mert $a$ első eleme pozitív: $\sigma = -\text{sgn}(a_1) \cdot \|a\|_2$.

In [ ]:
a = np.array([2, -2, 1], dtype=float)
n = len(a)
e1 = np.zeros(n); e1[0] = 1.0

print(f"a = {a}")
norm_a = np.linalg.norm(a)
print(f"||a||₂ = √(2² + (-2)² + 1²) = √{int(np.dot(a, a))} = {norm_a:.4f}")

# σ = -sgn(a1) · ||a||₂
sigma = -np.sign(a[0]) * norm_a
print(f"\nσ = -sgn({a[0]:.0f}) · {norm_a:.4f} = -1 · {norm_a:.0f} = {sigma:.4f}")
print(f"b = σ·e1 = [{sigma:.0f}, 0, 0]ᵀ")

# v meghatározása
diff = a - sigma * e1
print(f"\na - σe1 = [{a[0]:.0f}, {a[1]:.0f}, {a[2]:.0f}]ᵀ - ({sigma:.0f})·[1, 0, 0]ᵀ = {diff}")
print(f"  (valójában csak az első elemén kellett műveletet végezni: {a[0]:.0f} - ({sigma:.0f}) = {diff[0]:.0f})")

norm_diff = np.linalg.norm(diff)
print(f"||a - σe1||₂ = √({diff[0]:.0f}² + ({diff[1]:.0f})² + {diff[2]:.0f}²) = √{int(np.dot(diff, diff))} = {norm_diff:.4f}")

v = diff / norm_diff
print(f"\nv = (a - σe1)/||a - σe1||₂ = 1/√30 · [{diff[0]:.0f}, {diff[1]:.0f}, {diff[2]:.0f}]ᵀ = {v}")

# Ellenőrzés
print("\n=== Ellenőrzés: H(v)·a = σe1? ===")
vTa = np.dot(v, a)
print(f"  vᵀa = {vTa:.4f}")
Ha = a - 2 * vTa * v
print(f"  H(v)·a = a - 2(vᵀa)v = {Ha}")
print(f"  σ·e1   = [{sigma:.0f}, 0, 0]ᵀ")
print(f"  H(v)·a = σ·e1? {np.allclose(Ha, sigma * e1)}")

In [ ]:
def householder_v_ab(a, b):
    """Householder v vektor: H(v)·a = b, ahol ||a||₂ = ||b||₂."""
    diff = a - b
    return diff / np.linalg.norm(diff)

def householder_v_to_e1(a):
    """Householder v vektor: H(v)·a = σ·e1 (stabil előjelválasztással).
    
    Visszatérés:
        v: normált Householder-vektor
        sigma: az előjeles hossz
    """
    n = len(a)
    sigma = -np.sign(a[0]) * np.linalg.norm(a)
    if a[0] == 0:
        sigma = -np.linalg.norm(a)
    e1 = np.zeros(n); e1[0] = 1.0
    diff = a - sigma * e1
    v = diff / np.linalg.norm(diff)
    return v, sigma

def householder_apply(v, x):
    """H(v)·x kiszámítása mátrix felépítése nélkül: x - 2v(vᵀx)."""
    return x - 2 * np.dot(v, x) * v

# Gyors teszt
v_test, s_test = householder_v_to_e1(np.array([2, -2, 1], dtype=float))
print(f"householder_v_to_e1([2,-2,1]) → v = {v_test}, σ = {s_test:.4f}")
print(f"H(v)·a = {householder_apply(v_test, np.array([2,-2,1], dtype=float))}")

---
## 5. Householder-transzformáció alkalmazásai

### Felső háromszög alakra hozás (LER megoldása)

$Ax = b$ megoldása Householder-transzformációkkal:

$$Ax = b \;\to\; H_1 Ax = H_1 b \;\to\; \cdots \;\to\; \underbrace{H_{n-1} \cdots H_1}_{} A \cdot x = \underbrace{H_{n-1} \cdots H_1}_{} b$$

$$R \cdot x = d \quad \to \quad \text{visszahelyettesítés}$$

A $k$. lépésben a $k$. oszlop főátló alatti elemeit nullázzuk ki egy Householder-transzformációval.

**Műveletigény LER megoldásra:** $\frac{4}{3}n^3 + \mathcal{O}(n^2)$

### QR-felbontás Householder-módszerrel

$$H_{n-1} \cdots H_2 \cdot H_1 \cdot A = R \quad \Longrightarrow \quad A = \underbrace{H_1 \cdot H_2 \cdots H_{n-1}}_{Q} \cdot R = Q \cdot R$$

Mivel $H_k$ ortogonális és szimmetrikus ($H_k^{-1} = H_k^\top = H_k$), $Q$ is ortogonális.

$Q$-t úgy állítjuk elő, hogy egy egységmátrixból indulva jobbról alkalmazzuk a $H_k$ transzformációkat:
$Q_0 = I$, $Q_k = Q_{k-1} \cdot H_k$, $Q = Q_{n-1}$.

**Műveletigény QR-felbontásra:** $\frac{8}{3}n^3 + \mathcal{O}(n^2)$

**Megjegyzés:** Ez kicsit több, mint a Gram–Schmidt ($2n^3$), viszont numerikusan **stabilabb**.

### Kidolgozott példa: QR-felbontás Householder-módszerrel (3×3)

$$
A = \begin{bmatrix} 1 & 1 & 0 \\ 1 & 0 & 1 \\ 0 & 1 & 1 \end{bmatrix}
$$

Lépésenként kinullázzuk az oszlopok főátló alatti elemeit Householder-transzformációkkal.

In [ ]:
A_hh = np.array([[1, 1, 0],
                 [1, 0, 1],
                 [0, 1, 1]], dtype=float)
n = A_hh.shape[0]
print_matrix(A_hh, "A =")

R = A_hh.copy()
Q = np.eye(n)

# === 1. lépés: 1. oszlop kinullázása a főátló alatt ===
print("=" * 55)
print("1. lépés: 1. oszlop kinullázása (a főátló alatt)")
print("=" * 55)

a1_col = R[:, 0]  # teljes 1. oszlop
print(f"  a1 (1. oszlop) = {a1_col}")

v1, sigma1 = householder_v_to_e1(a1_col)
print(f"  σ1 = -sgn({a1_col[0]:.0f})·||a1||₂ = {sigma1:.4f}")
print(f"  v1 = {v1}")

# H1 mátrix (demonstrációhoz felépítjük)
H1 = np.eye(n) - 2 * np.outer(v1, v1)
print_matrix(H1, "\n  H1 =")

R = H1 @ R
Q = Q @ H1  # Q = Q · H1 (jobbról szorzunk)
print_matrix(R, "  H1 · A =")

# === 2. lépés: 2. oszlop kinullázása (2. sortól lefelé) ===
print("=" * 55)
print("2. lépés: 2. oszlop kinullázása (jobb alsó 2×2 részen)")
print("=" * 55)

# A jobb alsó (n-1)×(n-1) = 2×2 részmátrix 1. oszlopával dolgozunk
a2_sub = R[1:, 1]
print(f"  a2 (jobb alsó rész 1. oszlopa) = {a2_sub}")

v2_sub, sigma2 = householder_v_to_e1(a2_sub)
print(f"  σ2 = {sigma2:.4f}")
print(f"  ṽ2 = {v2_sub}")

# Teljes méretű H2
v2_full = np.zeros(n)
v2_full[1:] = v2_sub
H2 = np.eye(n) - 2 * np.outer(v2_full, v2_full)
print_matrix(H2, "\n  H2 =")

R = H2 @ R
Q = Q @ H2
print_matrix(R, "  H2 · H1 · A = R =")

print("=" * 55)
print("Eredmény")
print("=" * 55)
print_matrix(Q, "Q =")
print_matrix(R, "R =")
print(f"Q ortogonális? Q^T Q = I? {np.allclose(Q.T @ Q, np.eye(n))}")
print(f"Q @ R = A? {np.allclose(Q @ R, A_hh)}")
print_matrix(Q @ R, "Q @ R =")

In [ ]:
def qr_householder(A):
    """QR-felbontás Householder-módszerrel.
    
    Paraméterek:
        A: n×n invertálható mátrix
    
    Visszatérés:
        Q: ortogonális mátrix
        R: felső háromszögmátrix (r_ii > 0 nem garantált!)
    """
    n = A.shape[0]
    R = A.copy().astype(float)
    Q = np.eye(n)
    
    for k in range(n - 1):
        # A k. oszlop k. sortól lefelé eső része
        a_sub = R[k:, k]
        
        # Householder v vektor a részvektorra
        v_sub, sigma = householder_v_to_e1(a_sub)
        
        # Teljes méretű v
        v_full = np.zeros(n)
        v_full[k:] = v_sub
        
        # H_k alkalmazása R-re és Q-ra
        Hk = np.eye(n) - 2 * np.outer(v_full, v_full)
        R = Hk @ R
        Q = Q @ Hk
    
    return Q, R

# Ellenőrzés
Q_hh, R_hh = qr_householder(A_hh)
print("qr_householder függvény ellenőrzése:")
print(f"  Q @ R = A? {np.allclose(Q_hh @ R_hh, A_hh)}")
print(f"  Q ortogonális? {np.allclose(Q_hh.T @ Q_hh, np.eye(3))}")
print_matrix(Q_hh, "\nQ =")
print_matrix(R_hh, "R =")

### LER megoldása QR-felbontással

Oldjuk meg az $Ax = b$ rendszert QR-felbontással, ahol $A$ a fenti 3×3 mátrix és $b = [1, 2, 3]^\top$!

$$Ax = b \;\Longleftrightarrow\; QRx = b \;\Longleftrightarrow\; Rx = Q^\top b =: y$$

In [ ]:
b_qr = np.array([1, 2, 3], dtype=float)

print("Ax = b megoldása QR-felbontással")
print("=" * 55)
print(f"b = {b_qr}")

# 1. lépés: y = Q^T b
y = Q_hh.T @ b_qr
print(f"\n1. lépés: y = Qᵀb = {y}")

# 2. lépés: Rx = y visszahelyettesítés
print(f"\n2. lépés: Rx = y visszahelyettesítés")
print_matrix(R_hh, "R =")
print_vector(y, "y")

x_qr = np.zeros(3)
for i in range(2, -1, -1):
    x_qr[i] = (y[i] - np.dot(R_hh[i, i+1:], x_qr[i+1:])) / R_hh[i, i]
    print(f"  x{i+1} = (y{i+1} - Σ r_{i+1}j·xj) / r_{i+1}{i+1} = {x_qr[i]:.4f}")

print(f"\nMegoldás: x = {x_qr}")
print(f"Ellenőrzés: A @ x = {A_hh @ x_qr}")
print(f"A @ x = b? {np.allclose(A_hh @ x_qr, b_qr)}")
print(f"numpy megoldás: x = {np.linalg.solve(A_hh, b_qr)}")

---
## 6. Önálló feladatok

Az alábbi feladatokat a fenti példák mintájára oldják meg! Először próbálják meg **kézzel** (papír-ceruza), majd ellenőrizzék Pythonnal!

### 1. feladat: Ortogonalitás ellenőrzése

Döntsük el az alábbi mátrixokról, hogy ortogonálisak-e! Ellenőrizzük a $Q^\top Q = I$ feltételt és a $\det(Q) = \pm 1$ tulajdonságot!

$$
Q_1 = \begin{bmatrix} 1 & 0 \\ 0 & -1 \end{bmatrix}, \quad
Q_2 = \frac{1}{3}\begin{bmatrix} 2 & -1 & 2 \\ 2 & 2 & -1 \\ -1 & 2 & 2 \end{bmatrix}, \quad
Q_3 = \begin{bmatrix} 1 & 1 \\ 0 & 1 \end{bmatrix}
$$

In [ ]:
# 1. feladat -- megoldás helye
Q1_f = np.array([[1, 0], [0, -1]], dtype=float)
Q2_f = (1/3) * np.array([[2, -1, 2], [2, 2, -1], [-1, 2, 2]], dtype=float)
Q3_f = np.array([[1, 1], [0, 1]], dtype=float)

# IDE ÍRD A MEGOLDÁST!


In [ ]:
# 1. feladat -- ellenőrzés
for name, M in [("Q1", Q1_f), ("Q2", Q2_f), ("Q3", Q3_f)]:
    n = M.shape[0]
    ortog = np.allclose(M.T @ M, np.eye(n))
    det = np.linalg.det(M)
    print(f"{name}: ortogonális={ortog}, det={det:.4f}")
    if ortog:
        print_matrix(M.T @ M, f"  {name}ᵀ{name} =")
    else:
        print_matrix(M.T @ M, f"  {name}ᵀ{name} (≠ I!) =")

### 2. feladat: Gram–Schmidt ortogonalizáció

Készítsük el a következő mátrix QR-felbontását Gram–Schmidt-ortogonalizációval! Írjuk ki lépésenként az $r_{jk}$, $s_k$, $q_k$ értékeket!

$$
A = \begin{bmatrix} 1 & -1 & 4 \\ 1 & 4 & -2 \\ 1 & 4 & 2 \\ 1 & -1 & 0 \end{bmatrix}
$$

*Tipp:* Téglalap alakú mátrixra is működik a Gram–Schmidt! (4×3 → Q: 4×3, R: 3×3)

In [ ]:
# 2. feladat -- megoldás helye
A_f2 = np.array([[1, -1, 4],
                 [1,  4, -2],
                 [1,  4,  2],
                 [1, -1,  0]], dtype=float)

# IDE ÍRD A MEGOLDÁST!


In [ ]:
# 2. feladat -- ellenőrzés
Q_f2, R_f2 = np.linalg.qr(A_f2)
print_matrix(Q_f2, "numpy Q =")
print_matrix(R_f2, "numpy R =")
print(f"Q @ R = A? {np.allclose(Q_f2 @ R_f2, A_f2)}")

### 3. feladat: Householder-tükrözés

Határozzuk meg a Householder $v$ vektort, amely az $a$ vektort $\sigma \cdot e_1$ alakúra hozza!

$$
a = \begin{bmatrix} 1 \\ 2 \\ 2 \end{bmatrix}
$$

Számítsuk ki $\sigma$-t, $v$-t, és ellenőrizzük, hogy $H(v) \cdot a = \sigma e_1$!

In [ ]:
# 3. feladat -- megoldás helye
a_f3 = np.array([1, 2, 2], dtype=float)

# IDE ÍRD A MEGOLDÁST!


In [ ]:
# 3. feladat -- ellenőrzés
v_f3, sigma_f3 = householder_v_to_e1(a_f3)
Ha_f3 = householder_apply(v_f3, a_f3)
print(f"σ = {sigma_f3:.4f}")
print(f"v = {v_f3}")
print(f"H(v)·a = {Ha_f3}")
print(f"σ·e1   = {sigma_f3 * np.array([1,0,0])}")
print(f"Helyes? {np.allclose(Ha_f3, sigma_f3 * np.array([1,0,0]))}")

### 4. feladat: QR-felbontás és LER megoldása

Oldjuk meg az $Ax = b$ rendszert QR-felbontással (Gram–Schmidt **vagy** Householder módszerrel)!

$$
A = \begin{bmatrix} 2 & 1 & 1 \\ 1 & 3 & 2 \\ 1 & 2 & 3 \end{bmatrix}, \quad b = \begin{bmatrix} 4 \\ 6 \\ 6 \end{bmatrix}
$$

Lépések:
1. Állítsuk elő a QR-felbontást
2. Számítsuk ki $y = Q^\top b$
3. Oldjuk meg $Rx = y$-t visszahelyettesítéssel

In [ ]:
# 4. feladat -- megoldás helye
A_f4 = np.array([[2, 1, 1],
                 [1, 3, 2],
                 [1, 2, 3]], dtype=float)
b_f4 = np.array([4, 6, 6], dtype=float)

# IDE ÍRD A MEGOLDÁST!


In [ ]:
# 4. feladat -- ellenőrzés
x_f4 = np.linalg.solve(A_f4, b_f4)
print(f"numpy megoldás: x = {x_f4}")
print(f"A @ x = {A_f4 @ x_f4}")
print(f"A @ x = b? {np.allclose(A_f4 @ x_f4, b_f4)}")

### 5. feladat: Futásidő összehasonlítás

Hasonlítsuk össze a Gram–Schmidt, a Householder QR és a `numpy.linalg.qr` futásidejét különböző $n$ értékekre! Készítsünk grafikont!

*Tipp:* Használjunk véletlenszerű mátrixokat `np.random.randn(n, n)` segítségével.

In [ ]:
# 5. feladat -- megoldás helye
import matplotlib.pyplot as plt

n_values = [10, 30, 50, 100, 200]
time_gs = []
time_hh = []
time_np = []

for n in n_values:
    A_t = np.random.randn(n, n)
    
    # Gram-Schmidt
    t0 = time.perf_counter()
    for _ in range(3):
        gram_schmidt(A_t)
    t_gs = (time.perf_counter() - t0) / 3
    time_gs.append(t_gs)
    
    # Householder
    t0 = time.perf_counter()
    for _ in range(3):
        qr_householder(A_t)
    t_hh = (time.perf_counter() - t0) / 3
    time_hh.append(t_hh)
    
    # numpy
    t0 = time.perf_counter()
    for _ in range(3):
        np.linalg.qr(A_t)
    t_np = (time.perf_counter() - t0) / 3
    time_np.append(t_np)
    
    print(f"n={n:4d}: GS={t_gs:.4f}s, HH={t_hh:.4f}s, numpy={t_np:.6f}s")

plt.figure(figsize=(8, 5))
plt.plot(n_values, time_gs, 'o-', label='Gram-Schmidt')
plt.plot(n_values, time_hh, 's-', label='Householder')
plt.plot(n_values, time_np, '^-', label='numpy.linalg.qr')
plt.xlabel('n (mátrix mérete)')
plt.ylabel('Futásidő (s)')
plt.title('QR-felbontás módszerek futásideje')
plt.legend()
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.xscale('log')
plt.tight_layout()
plt.show()

---
## 7. Debuggolási feladatok

Az alábbi kódokban **szándékos hibák** vannak. Keressük meg és javítsuk ki őket!

### Debug 1: Hibás Gram–Schmidt implementáció

Az alábbi Gram–Schmidt implementáció **hibás eredményt** ad. Keressük meg a hibát!

*Tipp:* A hiba a projekciós lépésben van. Figyelj, melyik vektort szorozzuk be!

In [ ]:
def gram_schmidt_hibas(A):
    """HIBÁS Gram–Schmidt -- keresd meg a hibát!"""
    n = A.shape[0]
    Q = np.zeros((n, n))
    R = np.zeros((n, n))
    
    for k in range(n):
        for j in range(k):
            R[j, k] = np.dot(A[:, k], A[:, j])  # <-- VIZSGÁLD MEG EZT A SORT!
        
        s = A[:, k] - Q[:, :k] @ R[:k, k]
        R[k, k] = np.linalg.norm(s)
        Q[:, k] = s / R[k, k]
    
    return Q, R

# Tesztelés
A_test = np.array([[1, 1, 0],
                   [1, 0, 1],
                   [0, 1, 1]], dtype=float)

Q_hibas, R_hibas = gram_schmidt_hibas(A_test)
Q_helyes, R_helyes = gram_schmidt(A_test)

print(f"Hibás:  Q @ R = A? {np.allclose(Q_hibas @ R_hibas, A_test)}")
print(f"Helyes: Q @ R = A? {np.allclose(Q_helyes @ R_helyes, A_test)}")
print(f"\nHibás Q ortogonális?  {np.allclose(Q_hibas.T @ Q_hibas, np.eye(3))}")
print(f"Helyes Q ortogonális? {np.allclose(Q_helyes.T @ Q_helyes, np.eye(3))}")
print("\n>>> Keresd meg és javítsd ki a hibát a gram_schmidt_hibas függvényben!")
print(">>> Tipp: r_jk = ⟨a_k, q_j⟩, nem ⟨a_k, a_j⟩!")

### Debug 2: Hibás Householder v vektor

Az alábbi Householder implementáció hibás $v$ vektort állít elő. Keresd meg a hibát!

*Tipp:* A hiba a normálásban van. Mi a $H(v)$ definíciója? Mi kell $v$-ről teljesüljön?

In [ ]:
def householder_to_e1_hibas(a):
    """HIBÁS Householder v vektor -- keresd meg a hibát!"""
    n = len(a)
    sigma = -np.sign(a[0]) * np.linalg.norm(a)
    e1 = np.zeros(n); e1[0] = 1.0
    diff = a - sigma * e1
    v = diff  # <-- VIZSGÁLD MEG EZT A SORT! (normálás?)
    return v, sigma

# Tesztelés
a_test = np.array([2, -2, 1], dtype=float)

v_hibas, sigma_hibas = householder_to_e1_hibas(a_test)
v_helyes, sigma_helyes = householder_v_to_e1(a_test)

# H(v)·a kiszámítása a hibás v-vel
Ha_hibas = a_test - 2 * np.dot(v_hibas, a_test) * v_hibas
Ha_helyes = householder_apply(v_helyes, a_test)

print(f"Hibás v:  {v_hibas}   ||v|| = {np.linalg.norm(v_hibas):.4f}")
print(f"Helyes v: {v_helyes}   ||v|| = {np.linalg.norm(v_helyes):.4f}")
print(f"\nHibás:  H(v)·a = {Ha_hibas}  (kellene: [{sigma_hibas:.0f}, 0, 0])")
print(f"Helyes: H(v)·a = {Ha_helyes}  (kellene: [{sigma_helyes:.0f}, 0, 0])")
print(f"\n>>> Keresd meg a hibát! Mi a H(v) definíciójában v-re vonatkozó feltétel?")

### Debug 3: Hibás QR-felbontás Householder-rel

Az alábbi QR-felbontás implementáció hibás. A $Q$ mátrix előállítása nem helyes. Keresd meg a hibát!

*Tipp:* Gondolj arra, milyen sorrendben kell $H_k$-kat alkalmazni $Q$ előállításához: $Q = H_1 \cdot H_2 \cdots H_{n-1}$.

In [ ]:
def qr_householder_hibas(A):
    """HIBÁS QR Householder -- keresd meg a hibát!"""
    n = A.shape[0]
    R = A.copy().astype(float)
    Q = np.eye(n)
    
    for k in range(n - 1):
        a_sub = R[k:, k]
        v_sub, sigma = householder_v_to_e1(a_sub)
        
        v_full = np.zeros(n)
        v_full[k:] = v_sub
        
        Hk = np.eye(n) - 2 * np.outer(v_full, v_full)
        R = Hk @ R
        Q = Hk @ Q  # <-- VIZSGÁLD MEG EZT A SORT!
    
    return Q, R

# Tesztelés
A_test = np.array([[1, 1, 0],
                   [1, 0, 1],
                   [0, 1, 1]], dtype=float)

Q_hibas, R_hibas = qr_householder_hibas(A_test)
Q_helyes, R_helyes = qr_householder(A_test)

print(f"Hibás:  Q @ R = A? {np.allclose(Q_hibas @ R_hibas, A_test)}")
print(f"Helyes: Q @ R = A? {np.allclose(Q_helyes @ R_helyes, A_test)}")
print(f"\nHibás  Q ortogonális? {np.allclose(Q_hibas.T @ Q_hibas, np.eye(3))}")
print(f"Helyes Q ortogonális? {np.allclose(Q_helyes.T @ Q_helyes, np.eye(3))}")

print(f"\nHibás  Q @ R =")
print_matrix(Q_hibas @ R_hibas)
print(f"Helyes Q @ R =")
print_matrix(Q_helyes @ R_helyes)

print(">>> Keresd meg a hibát! Q = H1·H2·...·H_{n-1}, tehát Q-t jobbról szorozzuk Hk-val!")
print(">>> A hibás kód: Q = Hk @ Q (balról szoroz), a helyes: Q = Q @ Hk (jobbról szoroz)")

---
## Összefoglaló

| Téma | Lényeg |
|------|--------|
| **Ortogonális mátrix** | $Q^\top Q = I$; oszlopai ortonormált rendszert alkotnak; hosszmegőrző |
| **QR-felbontás** | $A = QR$: $Q$ ortogonális, $R$ felső háromszög; létezik ha $\det A \neq 0$ |
| **Gram–Schmidt** | Ortogonalizációs eljárás → QR-felbontás; műveletigény: $2n^3 + \mathcal{O}(n^2)$ |
| **Normálás nélkül** | Kézi számolásra alkalmasabb; utólag normálunk |
| **Householder-mátrix** | $H(v) = I - 2vv^\top$; szimmetrikus, ortogonális, tükröző ($H^2 = I$) |
| **Householder-transzformáció** | $H(v)x = x - 2v(v^\top x)$: csak $4n$ művelet, nem kell $H$-t felépíteni |
| **Householder → $\sigma e_1$** | $\sigma = -\text{sgn}(a_1)\|a\|_2$; $v = (a - \sigma e_1)/\|a - \sigma e_1\|_2$ |
| **LER megoldása HH-val** | Felső háromszög alakra hozás + visszahelyettesítés; $\frac{4}{3}n^3 + \mathcal{O}(n^2)$ |
| **QR-felbontás HH-val** | $n-1$ db Householder-transzformáció; $\frac{8}{3}n^3 + \mathcal{O}(n^2)$; **stabilabb** mint GS |